In [1]:
## Semantic Chunking

"""
Semantic Junker is a document splitter that uses embedding similarity between sentences to decide junk boundaries.
It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character or token splitters. """

'\nSemantic Junker is a document splitter that uses embedding similarity between sentences to decide junk boundaries.\nIt ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character or token splitters. '

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [5]:
## Initialize the model
model = SentenceTransformer("all-MiniLM-L6-v2")


## Sample text
text ="""
Langchain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecon.
You can create Chains, Agents, Memory, and Retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

## Step 1: Split into sentences

sentences = [s.strip() for s in text.split("\n") if s.strip()]


## Step 2: Embed each sentence
embeddings = model.encode(sentences)

## Step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk = [sentences[0]]

## Step 4: Semantic grouping based on threshold
for i in range(1, len(sentences)):
    sim = cosine_similarity(
        embeddings[i - 1 : i],
        embeddings[i : i + 1],
    )[0, 0]


    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

## Append the last chunk
chunks.append(" ".join(current_chunk))


## Output the chunks
print("\n Semantic Chunks:")
for i,chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: \n{chunk}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7783.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Semantic Chunks:

Chunk 1: 
Langchain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecon.

Chunk 2: 
You can create Chains, Agents, Memory, and Retrievers.

Chunk 3: 
The Eiffel Tower is located in Paris.

Chunk 4: 
France is a popular tourist destination.


In [6]:
### RAG Pipeline Modular Coding

In [8]:
import os

from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_openai import OpenAIEmbeddings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

if os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "").strip()

In [12]:
## Custom Semantic Chunker with Threshold

class ThresholdSemanticChunker:
    def __init__(self, model_name = "all-MiniLM-L6-v2", threshold = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold=threshold


    def split(self,text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(".".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
            
            chunks.append(".".join(current_chunk) +".")
            return chunks

    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content = chunk, metadata= doc.metadata))
        return result

In [13]:
# Sample Text
sample_text = """
Langchain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecon.
You can create Chains, Agents, Memory, and Retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
""" 


doc = Document(page_content = sample_text)
doc

Document(metadata={}, page_content='\nLangchain is a framework for building applications with LLMs.\nLangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecon.\nYou can create Chains, Agents, Memory, and Retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [14]:
## Chunking
chunker = ThresholdSemanticChunker(threshold = 0.7)
chunks = chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9362.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={}, page_content='Langchain is a framework for building applications with LLMs.LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecon.')]

In [15]:
### VectorStore
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "").strip()
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks,embedding)
retriever = vectorstore.as_retriever()

In [16]:
## Prompt Template

#  --- 5. Prompt Template ---
template = """ Answer the question based on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=' Answer the question based on the following context:\n{context}\n\nQuestion: {question}\n')

In [19]:
## LLM
llm = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq", temperature=0.4)

### LCEL chain with retrieval (close RunnableMap before `|`)

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"],
        },
    )
    | prompt
    | llm
    | StrOutputParser()
)

query = {"question": "What is Langchain used for?"}
result = rag_chain.invoke(query)

print(result)

Langchain is a framework for building applications with Large Language Models (LLMs).


In [18]:
## Semantic chunker with Langchain

In [21]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

ModuleNotFoundError: No module named 'langchain.document_loaders'

In [ ]:
## Load the documents
loader  = TextLoader("langchain_intro.txt")
loader.load()

## I